In [11]:
import pandas as pd

df = pd.read_parquet("../data/input/yellow_tripdata_2026-01.parquet")
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [12]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
df.columns.tolist()

Rows: 3724889
Columns: 20


['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3724889 entries, 0 to 3724888
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [14]:
df.isna().sum().sort_values(ascending=False)

passenger_count          1088058
congestion_surcharge     1088058
store_and_fwd_flag       1088058
RatecodeID               1088058
Airport_fee              1088058
tpep_dropoff_datetime          0
VendorID                       0
tpep_pickup_datetime           0
DOLocationID                   0
payment_type                   0
trip_distance                  0
PULocationID                   0
extra                          0
fare_amount                    0
mta_tax                        0
tip_amount                     0
improvement_surcharge          0
tolls_amount                   0
total_amount                   0
cbd_congestion_fee             0
dtype: int64

In [15]:
df[[
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "tip_amount",
    "tolls_amount"
]].describe()

,passenger_count,trip_distance,fare_amount,total_amount,tip_amount,tolls_amount
count,2.636831e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06,3.724889e+06
mean,1.256271e+00,6.455647e+00,2.080425e+01,2.917853e+01,2.608142e+00,4.983751e-01
std,6.702431e-01,6.488855e+02,1.892701e+01,2.258553e+01,3.917405e+00,2.131731e+00
min,0.000000e+00,0.000000e+00,-2.555200e+03,-2.560200e+03,-8.888000e+01,-9.450000e+01
25%,1.000000e+00,1.000000e+00,1.000000e+01,1.700000e+01,0.000000e+00,0.000000e+00
50%,1.000000e+00,1.810000e+00,1.560000e+01,2.305000e+01,2.000000e+00,0.000000e+00
75%,1.000000e+00,3.730000e+00,2.610000e+01,3.383000e+01,3.710000e+00,0.000000e+00
max,9.000000e+00,2.690975e+05,2.555200e+03,2.560200e+03,7.660000e+02,1.222200e+02


In [16]:
bad_dates = df[df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]]
len(bad_dates)

1

In [17]:
df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["average_speed_mph"] = df["trip_distance"] / (df["trip_duration_minutes"] / 60)

df["pickup_year"] = df["tpep_pickup_datetime"].dt.year
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month

df["revenue_per_mile"] = df["total_amount"] / df["trip_distance"]

df[[
    "trip_duration_minutes",
    "average_speed_mph",
    "pickup_year",
    "pickup_month",
    "revenue_per_mile"
]].head()

,trip_duration_minutes,average_speed_mph,pickup_year,pickup_month,revenue_per_mile
0,5.550000,10.486486,2026,1,16.350515
1,5.716667,9.446064,2026,1,15.166667
2,8.883333,9.455910,2026,1,13.535714
3,42.800000,7.822430,2026,1,9.956989
4,13.500000,9.600000,2026,1,10.694444


In [18]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.validators.taxi_validator import TaxiValidator

validator = TaxiValidator()
validator.validate(df)

VALIDATION ERRORS:
- passenger_count has 1088058 null values
- 1 rows have dropoff before pickup
- fare_amount has 39463 negative values
- total_amount has 39984 negative values


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_duration_minutes,average_speed_mph,pickup_year,pickup_month,revenue_per_mile
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,1.0,15.86,2.5,0.0,0.00,5.550000,10.486486,2026,1,16.350515
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,1.0,13.65,2.5,0.0,0.75,5.716667,9.446064,2026,1,15.166667
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,1.0,18.95,2.5,0.0,0.75,8.883333,9.455910,2026,1,13.535714
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,1.0,55.56,2.5,0.0,0.75,42.800000,7.822430,2026,1,9.956989
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,1.0,23.10,2.5,0.0,0.75,13.500000,9.600000,2026,1,10.694444
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,None,237,161,0,...,1.0,21.84,NaN,NaN,0.75,13.266667,7.326633,2026,1,13.481481
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,None,42,42,0,...,1.0,24.69,NaN,NaN,0.00,0.233333,0.000000,2026,1,inf
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,None,137,69,0,...,1.0,33.96,NaN,NaN,0.75,15.783333,26.002112,2026,1,4.964912
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,None,137,162,0,...,1.0,23.94,NaN,NaN,0.75,9.650000,9.512953,2026,1,15.647059


In [19]:
from src.processors.taxi_processor import TaxiProcessor

processor = TaxiProcessor()
processed_df = processor.process(df)

processed_df.head()
processed_df.columns.tolist()
processed_df[[
    "trip_duration_minutes",
    "average_speed_mph",
    "pickup_year",
    "pickup_month",
    "revenue_per_mile",
    "trip_distance_category",
    "fare_category",
    "trip_time_of_day"
]].head()

,trip_duration_minutes,average_speed_mph,pickup_year,pickup_month,revenue_per_mile,trip_distance_category,fare_category,trip_time_of_day
0,5.550000,10.486486,2026,1,16.350515,Short,Low,Night
1,5.716667,9.446064,2026,1,15.166667,Short,Low,Night
2,8.883333,9.455910,2026,1,13.535714,Short,Low,Night
3,42.800000,7.822430,2026,1,9.956989,Medium,Medium,Night
4,13.500000,9.600000,2026,1,10.694444,Medium,Low,Night


In [21]:
import sys
import os
import importlib

sys.path.insert(0, os.path.abspath(".."))

# remove cached version
if "src.validators.backup_validator" in sys.modules:
    del sys.modules["src.validators.backup_validator"]

importlib.invalidate_caches()

from src.validators.backup_validator import BackupValidator

backup_validator = BackupValidator()
backup_validator.validate(processed_df)

BACKUP VALIDATION ERRORS:
- 1 rows have negative trip duration


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,...,Airport_fee,cbd_congestion_fee,trip_duration_minutes,average_speed_mph,pickup_year,pickup_month,revenue_per_mile,trip_distance_category,fare_category,trip_time_of_day
0,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,239,238,1,7.20,1.00,0.5,...,0.0,0.00,5.550000,10.486486,2026,1,16.350515,Short,Low,Night
1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,163,162,2,7.90,4.25,0.5,...,0.0,0.75,5.716667,9.446064,2026,1,15.166667,Short,Low,Night
2,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,43,237,1,10.70,4.25,0.5,...,0.0,0.75,8.883333,9.455910,2026,1,13.535714,Short,Low,Night
3,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,142,209,1,38.70,1.00,0.5,...,0.0,0.75,42.800000,7.822430,2026,1,9.956989,Medium,Medium,Night
4,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,88,144,1,13.50,1.00,0.5,...,0.0,0.75,13.500000,9.600000,2026,1,10.694444,Medium,Low,Night
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,237,161,0,17.09,0.00,0.5,...,NaN,0.75,13.266667,7.326633,2026,1,13.481481,Short,Low,Evening
3724885,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,42,42,0,23.19,0.00,0.5,...,NaN,0.00,0.233333,0.000000,2026,1,NaN,Short,Medium,Evening
3724886,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,137,69,0,29.21,0.00,0.5,...,NaN,0.75,15.783333,26.002112,2026,1,4.964912,Medium,Medium,Evening
3724887,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,137,162,0,19.19,0.00,0.5,...,NaN,0.75,9.650000,9.512953,2026,1,15.647059,Short,Low,Evening


In [22]:
from src.writers.local_writer import LocalWriter

writer = LocalWriter()

writer.write_parquet(
    processed_df,
    "../data/output/processed_taxi_2026_01.parquet"
)

Data written to ../data/output/processed_taxi_2026_01.parquet
